This cell initializes the analysis by loading pre-computed data artifacts generated from a previous processing step, ensuring that the current notebook can operate without needing to re-run computationally intensive tasks like API calls or clustering algorithms. By retrieving structured data on concerns, benefits, and their embeddings, along with cluster summaries and mappings, it sets the foundation for subsequent analyses of shared concerns and benefits across public dialogue documents, which is crucial for understanding the framing and thematic patterns in public discourse around technology.

In [ ]:
# @title Load pre-computed artifacts
# Run this cell before any analysis cell.  It loads all outputs written by
# 01_processing.ipynb from disk so this notebook never calls the OpenAI API
# or re-runs k-means.
from pathlib import Path
import pub_dialogue.utils as du
from pub_dialogue.utils import show_status, show_complete, AccessStage, AddressStage
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- CIP-0010: stage objects centralise all path / parameter constants ---
_access  = AccessStage()
_address = AddressStage(access=_access)
OUTPUT_FOLDER     = _access.output_folder
CHECKPOINT_FOLDER = _access.checkpoint_folder
TECH_COL          = _address.tech_col

a = _access.load_artifacts()

chunks_df    = a["chunks_df"]
concerns_df  = a["concerns_df"]
benefits_df  = a["benefits_df"]

# extracted_concerns/benefits.csv are saved before the technology_meta merge
# in 01_processing.ipynb, so we re-apply it here if needed.
if "technology_meta" not in concerns_df.columns:
    _tech = chunks_df[["chunk_id", "technology_meta"]]
    concerns_df = concerns_df.merge(_tech, on="chunk_id", how="left")
    benefits_df = benefits_df.merge(_tech, on="chunk_id", how="left")

concern_embeddings  = a["concern_embeddings"]
benefit_embeddings  = a["benefit_embeddings"]
concern_centroids   = a["concern_centroids"]
benefit_centroids   = a["benefit_centroids"]

concern_ids          = a["concern_ids"]
benefit_ids          = a["benefit_ids"]
cluster_labels       = a["cluster_labels"]
benefit_cluster_labels = a["benefit_cluster_labels"]
cluster_summary_df   = a["cluster_summary_df"]
benefit_cluster_summary_df = a["benefit_cluster_summary_df"]

framing_lens_mappings         = a["framing_lens_mappings"]
benefit_framing_lens_mappings = a["benefit_framing_lens_mappings"]

cluster_entropy           = a["cluster_entropy"]
cluster_entropy_norm      = a["cluster_entropy_norm"]
cross_cutting_clusters    = a["cross_cutting_clusters"]

benefit_cluster_entropy          = a["benefit_cluster_entropy"]
normalized_entropy_benefits      = a["normalized_entropy_benefits"]
cross_cutting_clusters_benefits  = a["cross_cutting_clusters_benefits"]

# Convenience: CLUSTER_LABELS / BENEFIT_CLUSTER_LABELS dicts used by plots
CLUSTER_LABELS        = {int(k): v for k, v in cluster_labels.items()}
BENEFIT_CLUSTER_LABELS = {int(k): v for k, v in benefit_cluster_labels.items()}

# Convenience aliases: uppercase names used by analysis cells
FRAMING_LENS_MAPPINGS        = framing_lens_mappings
BENEFIT_FRAMING_LENS_MAPPINGS = benefit_framing_lens_mappings

# Aliases for variable names used in analysis cells
centroids            = concern_centroids
cluster_labels_dict        = cluster_labels
benefit_cluster_labels_dict = benefit_cluster_labels

print(f"Artifacts loaded from {OUTPUT_FOLDER} / {CHECKPOINT_FOLDER}")
print(f"  Chunks: {len(chunks_df):,}  |  Concerns: {len(concerns_df):,}  |  Benefits: {len(benefits_df):,}")

This cell calculates the prevalence of various framing lenses across the analyzed public dialogue documents by aggregating the number of concern phrases associated with each lens. This information is crucial for understanding how different perspectives shape public discourse on technology, allowing researchers to identify dominant themes and their potential influence on public opinion and policy-making. The resulting visualization provides a clear overview of the relative importance of each framing lens, facilitating deeper insights into the shared concerns and benefits expressed in the dialogues.

In [ ]:
# @title Compute framing lens prevalence

show_status("Computing framing lens prevalence...")

lens_prevalence = {}
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    lens_mask = concerns_df['cluster_id'].isin(data['cluster_ids'])
    lens_prevalence[lens_name] = lens_mask.sum()

lens_prev_df = pd.DataFrame([
    {'Framing Lens': k, 'Concerns': v}
    for k, v in lens_prevalence.items()
]).sort_values('Concerns', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(lens_prev_df['Framing Lens'], lens_prev_df['Concerns'], color='steelblue')
ax.set_xlabel('Number of Concern Phrases')
ax.set_title('Framing Lens Prevalence Across All Documents')

for bar, val in zip(bars, lens_prev_df['Concerns']):
    ax.text(val + 10, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "framing_lens_prevalence.png", dpi=150, bbox_inches='tight')
plt.show()

show_complete("Framing lens prevalence computed")

This cell generates a dendrogram to visualize the hierarchical relationships among concern clusters identified in the public dialogue documents, validating the framing lens scheme by assessing whether clusters grouped by a language model align with a data-driven hierarchy. The strong correspondence or divergence observed in this analysis is crucial, as it informs the robustness of the framing lens framework and highlights the underlying structure of shared concerns across technologies, ultimately enhancing the understanding of public sentiment and discourse in the context of emerging technologies.

In [ ]:
# @title Hierarchical structure of concern clusters (dendrogram)
# Validates the framing-lens scheme by showing whether concern clusters
# that the LLM grouped into the same lens also cluster together in a
# data-driven hierarchy. Strong correspondence supports the lens scheme;
# divergence is a finding worth reporting.

import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

show_status("Computing concern-cluster dendrogram...")

# L2-normalise centroids (cosine distance via euclidean on normed vectors)
_norm = np.linalg.norm(centroids, axis=1, keepdims=True)
centroids_normed = centroids / np.clip(_norm, 1e-12, None)

# Pairwise cosine distance and average linkage
distances = pdist(centroids_normed, metric="cosine")
Z = linkage(distances, method="average")

# Build leaf labels from existing cluster labels
cluster_label_lookup = {
    int(k): v.get("label", f"Cluster {k}")
    for k, v in cluster_labels_dict.items()
}
leaf_labels = [cluster_label_lookup.get(i, f"Cluster {i}")
               for i in range(len(centroids_normed))]

# Build cluster->lens lookup
cluster_to_lens = {}
for lens_name, info in FRAMING_LENS_MAPPINGS.items():
    for cid in info.get("cluster_ids", []):
        cluster_to_lens[int(cid)] = lens_name

# Assign a colour per lens
import matplotlib.cm as cm
lens_names = list(FRAMING_LENS_MAPPINGS.keys())
lens_colours = {name: cm.tab20(i / max(len(lens_names), 1))
                for i, name in enumerate(lens_names)}

# Plot
fig, ax = plt.subplots(figsize=(11, max(8, len(leaf_labels) * 0.18)))
dd = dendrogram(
    Z,
    labels=leaf_labels,
    orientation="left",
    leaf_font_size=8,
    color_threshold=0,
    above_threshold_color="grey",
    ax=ax,
)
# Recolour leaf labels by lens
ylabels = ax.get_ymajorticklabels()
for label in ylabels:
    cid_label = label.get_text()
    matched_cid = next((cid for cid, name in cluster_label_lookup.items()
                        if name == cid_label), None)
    if matched_cid is not None:
        lens = cluster_to_lens.get(matched_cid, None)
        if lens is not None:
            label.set_color(lens_colours.get(lens, "black"))

ax.set_xlabel("Cosine distance between concern cluster centroids")
ax.set_title("Hierarchical structure of concern clusters\n(leaf colour = LLM-assigned framing lens)")

from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=col, label=name) for name, col in lens_colours.items()]
ax.legend(handles=legend_handles, loc="lower right", fontsize=7, title="Framing lens")

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "concern_cluster_dendrogram.png", dpi=200, bbox_inches="tight")
plt.show()

# Quantitative comparison: ARI between LLM lens assignment and dendrogram cut
from sklearn.metrics import adjusted_rand_score

n_lenses = len(FRAMING_LENS_MAPPINGS)
data_driven_groups = fcluster(Z, t=n_lenses, criterion="maxclust")
lens_assignment = np.array([
    lens_names.index(cluster_to_lens.get(i, lens_names[0]))
    if cluster_to_lens.get(i) in lens_names else -1
    for i in range(len(centroids_normed))
])
mask = lens_assignment >= 0
ari = adjusted_rand_score(lens_assignment[mask], data_driven_groups[mask])
print(f"\nNumber of concern lenses (LLM-derived): {n_lenses}")
print(f"Adjusted Rand Index between lens assignment and dendrogram cut at "
      f"{n_lenses} groups: {ari:.3f}")
print("Higher = stronger correspondence between data-driven hierarchy and "
      "LLM-derived lens scheme.")

show_complete("Concern-cluster dendrogram complete.")

This cell computes the normalized entropy of concerns associated with different technologies in public dialogue documents, identifying cross-cutting themes that exhibit significant diversity in perspectives. By establishing a threshold for normalized entropy, it highlights clusters of concerns that are prevalent across multiple technologies, which is crucial for understanding shared societal values and priorities in the context of public dialogue on technology. This analysis aids in framing discussions around technology by revealing commonalities and divergences in public sentiment, ultimately informing policy and communication strategies.

In [ ]:
# --- Section 6 prerequisites (canonical tech + cross-cutting clusters) ---

import numpy as np

TECH_COL = "technology_meta"

# v16-fix: compute normalised entropy locally rather than relying on a global
# called `normalized_entropy` (which is a function in this notebook, not a dict).
# This mirrors the same fix in the stable-core figure cell.
n_techs_for_norm = concerns_df[TECH_COL].nunique()
max_ent_for_norm = np.log(n_techs_for_norm) if n_techs_for_norm > 1 else 1.0
_norm_ent_for_xcut = {cid: (e / max_ent_for_norm) for cid, e in cluster_entropy.items()}

CROSS_CUTTING_ENTROPY_THRESHOLD = 0.5
cross_cutting_cluster_ids = {
    cid for cid, ent in _norm_ent_for_xcut.items()
    if ent >= CROSS_CUTTING_ENTROPY_THRESHOLD
}

print(f"Cross-cutting clusters: {len(cross_cutting_cluster_ids)} of {len(cluster_entropy)} "
      f"(normalised entropy >= {CROSS_CUTTING_ENTROPY_THRESHOLD})")

This cell computes the shared concerns across various technologies by analyzing the prevalence and entropy of concern clusters in the dialogue documents. By creating a document-weighted salience matrix and identifying the top clusters with high prevalence and cross-technology spread, it highlights key issues that resonate broadly across different technological contexts, which is crucial for understanding public sentiment and guiding future dialogue initiatives. The resulting visualizations and data outputs facilitate further exploration of these shared concerns, informing policymakers and researchers about common themes in public discourse.

In [ ]:
# @title Shared concerns across technologies (document-weighted)

show_status("Computing shared concern structure (all technologies)...")

TECH_COL = "technology_meta"

# Canonical list of technologies from metadata
technologies = sorted(concerns_df[TECH_COL].dropna().unique().tolist())

# Salience matrix: rows=technology, cols=cluster_id, values=proportion of concerns
salience_rows = {}
for tech in technologies:
    tech_mask = concerns_df[TECH_COL] == tech
    tech_total = int(tech_mask.sum())
    if tech_total == 0:
        continue
    counts = concerns_df.loc[tech_mask, "cluster_id"].value_counts()
    salience_rows[tech] = (counts / tech_total).to_dict()

salience_df = pd.DataFrame(salience_rows).T.fillna(0)

# Document-weighted global prevalence of clusters
global_cluster_prevalence = concerns_df["cluster_id"].value_counts(normalize=True)

# Cross-cutting metric (entropy over technologies) computed earlier as cluster_entropy
cluster_meta_df = (
    pd.DataFrame(
        {
            "cluster_id": list(cluster_entropy.keys()),
            "tech_entropy": [cluster_entropy[c] for c in cluster_entropy.keys()],
            "global_prevalence": [global_cluster_prevalence.get(c, 0) for c in cluster_entropy.keys()],
        }
    )
    .set_index("cluster_id")
    .sort_values(["global_prevalence", "tech_entropy"], ascending=False)
)

# Shared concerns = high prevalence + high entropy
shared_concerns = cluster_meta_df.head(20)

print("Top shared concern clusters (high prevalence + cross-technology spread):")
display(shared_concerns.head(12))

# Quick bar chart (prevalence)
plt.figure(figsize=(10, 5))
shared_concerns.sort_values("global_prevalence")["global_prevalence"].plot(kind="barh", legend=False)
plt.title("Shared concerns across technologies (top 20 clusters by prevalence × spread)")
plt.xlabel("Share of all extracted concern phrases")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "shared_concerns_top20.png")
plt.show()

shared_concerns.reset_index().to_csv(OUTPUT_FOLDER / "shared_concerns_top20.csv", index=False)

show_complete("Shared concerns computed")


This cell computes and visualizes the prevalence and entropy of different framing lenses across various technologies in the public dialogue documents. By analyzing the distribution of concerns related to each lens, it identifies which framings are most commonly shared and how they vary across technologies, providing insights into collective public perceptions and concerns. This analysis is crucial for understanding cross-cutting themes in public dialogue, informing policy and communication strategies in technology development.

In [ ]:
# @title Shared framings across technologies (document-weighted)

from scipy.stats import entropy

show_status("Computing shared framing structure (all technologies)...")

TECH_COL = 'technology_meta'
technologies = sorted(concerns_df[TECH_COL].dropna().unique().tolist())

# Lens prevalence overall + entropy across technologies
lens_stats = []
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    lens_clusters = set(data['cluster_ids'])
    lens_mask = concerns_df['cluster_id'].isin(lens_clusters)
    overall_prev = lens_mask.mean()  # document-weighted (each concern phrase counts equally)

    # Technology distribution within lens
    tech_counts = concerns_df.loc[lens_mask, TECH_COL].value_counts()
    tech_probs = (tech_counts / tech_counts.sum()).values if tech_counts.sum() > 0 else np.array([])
    lens_entropy = float(entropy(tech_probs)) if len(tech_probs) > 1 else 0.0

    lens_stats.append({
        'framing_lens': lens_name,
        'overall_prevalence': float(overall_prev),
        'tech_entropy': lens_entropy,
        'n_concerns_in_lens': int(lens_mask.sum())
    })

lens_meta_df = (pd.DataFrame(lens_stats)
                .sort_values(['overall_prevalence','tech_entropy'], ascending=False))

print("Shared framings (high prevalence + cross-technology spread):")
display(lens_meta_df.head(12))

# Scatter: prevalence vs entropy
plt.figure(figsize=(7,5))
plt.scatter(lens_meta_df['tech_entropy'], lens_meta_df['overall_prevalence'])
for _, r in lens_meta_df.iterrows():
    plt.text(r['tech_entropy'], r['overall_prevalence'], r['framing_lens'], fontsize=8)
plt.xlabel("Entropy across technologies")
plt.ylabel("Share of all extracted concerns")
plt.title("Shared framings across technologies")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "shared_framings_scatter.png")
plt.show()

lens_meta_df.to_csv(OUTPUT_FOLDER / "shared_framings.csv", index=False)
show_complete("Shared framings computed")


This cell visualizes the stable core of public concerns regarding various technologies by plotting the relationship between the normalized entropy of concern clusters and their frequency. By identifying clusters that are both cross-cutting and frequently mentioned, the analysis highlights key areas of shared public interest, which can inform policymakers and stakeholders about prevalent societal concerns and facilitate more targeted dialogue in public engagement efforts.

In [ ]:
# @title Visualise the stable core of public concerns

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# ---- prerequisites ----
if "cluster_entropy" not in globals():
    raise NameError("cluster_entropy not found. Run the cluster entropy section first.")
if "concerns_df" not in globals():
    raise NameError("concerns_df not found. Run concern extraction first.")

TECH_COL = "technology_meta"
ENTROPY_THRESHOLD = 0.5  # canonical cross-cutting threshold (applied to NORMALISED entropy)

# ---- build cluster-level dataframe ----
# v16-fix: my earlier patch assumed `normalized_entropy` was a dict, but it's
# actually a function defined elsewhere in the notebook. Compute normalised
# entropy locally from the raw `cluster_entropy` dict instead.
n_techs = concerns_df[TECH_COL].nunique()
max_entropy = np.log(n_techs) if n_techs > 1 else 1.0
norm_ent_dict = {cid: (e / max_entropy) for cid, e in cluster_entropy.items()}

# cluster size = number of concern phrases
cluster_size = concerns_df["cluster_id"].value_counts()

df = (
    pd.DataFrame({
        "cluster_id": list(norm_ent_dict.keys()),
        "entropy": [norm_ent_dict[c] for c in norm_ent_dict.keys()],
    })
    .set_index("cluster_id")
)
df["size"] = cluster_size
df["size"] = df["size"].fillna(0)
df = df[df["size"] > 0]

# cluster labels (if available)
if "CLUSTER_LABELS" in globals():
    df["label"] = df.index.map(lambda c: CLUSTER_LABELS.get(c, f"Cluster {c}"))
else:
    df["label"] = df.index.map(lambda c: f"Cluster {c}")

entropy_vals = df["entropy"].to_numpy(dtype=float)
size = df["size"].to_numpy(dtype=float)
labels = df["label"].to_numpy(dtype=str)

# ---- define cross-cutting + stable core ----
cross_cutting = entropy_vals >= ENTROPY_THRESHOLD
size_thresh = np.quantile(size, 0.75)  # top 25% by frequency
stable_core = cross_cutting & (size >= size_thresh)

# clusters to annotate (most "core")
score = entropy_vals * np.log1p(size)
annotate_idx = np.argsort(score)[::-1][:12]

# ---- plot ----
plt.figure(figsize=(10, 7))
ax = plt.gca()

# shaded stable-core region
core_region = Rectangle(
    (ENTROPY_THRESHOLD, size_thresh),
    1.0 - ENTROPY_THRESHOLD,
    size.max() * 1.05 - size_thresh,
    alpha=0.12,
    zorder=0
)
ax.add_patch(core_region)
ax.text(
    ENTROPY_THRESHOLD + 0.01,
    size.max() * 1.02,
    "Stable core\n(cross-cutting + high frequency)",
    fontsize=10,
    va="top"
)

plt.scatter(entropy_vals[~cross_cutting], size[~cross_cutting], alpha=0.6, s=35, label="More tech-specific clusters")
plt.scatter(entropy_vals[cross_cutting], size[cross_cutting], alpha=0.8, s=45, label="More cross-cutting clusters")
plt.scatter(entropy_vals[stable_core], size[stable_core], s=90, label="Stable core")

for i in annotate_idx:
    plt.annotate(labels[i], (entropy_vals[i], size[i]), textcoords="offset points", xytext=(6, 4), ha="left", fontsize=9)

plt.axvline(ENTROPY_THRESHOLD, linestyle="--", linewidth=1)
plt.axhline(size_thresh, linestyle="--", linewidth=1)
plt.xlabel("Technology-distribution entropy (higher = more cross-cutting)")
plt.ylabel("Cluster frequency / size (higher = more recurrent)")
plt.title("Figure: The Stable Core of Public Technology Concerns")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "concern_stable_core_scatter.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

This cell visualizes the embedding space of concerns related to various technologies by projecting high-dimensional salience vectors into a two-dimensional space using PCA. The resulting scatter plot allows for the identification of clusters of concerns, with point sizes indicating cluster frequency and colors representing different framing lenses, thereby facilitating a nuanced understanding of how public dialogue reflects shared concerns and benefits across technologies. This visualization is crucial for interpreting the underlying patterns in public sentiment and framing, which can inform future dialogue strategies and policy-making.

In [ ]:
# @title Visualise the concern embedding space
# Clusters projected into 2D using PCA over technology-salience vectors.
# Point size ~ cluster frequency proxy; color ~ framing lens (if available).

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ARTIFACT_DIR = globals().get("ARTIFACT_DIR", "analysis_artifacts")
EXCEL_PATH = globals().get("EXCEL_PATH", os.path.join(ARTIFACT_DIR, "public_dialogue_analysis_v9.xlsx"))

# 1) Load tech x cluster salience (in-memory)
if "salience_df" not in globals():
    raise NameError("salience_df not found. Run Section 6.1a to compute salience_df first.")

tech_by_cluster = salience_df.copy()
tech_by_cluster = tech_by_cluster.apply(pd.to_numeric, errors="coerce").fillna(0)

# Convert to clusters x technologies
cluster_features = tech_by_cluster.T  # rows=clusters, cols=technologies
clusters = cluster_features.index.astype(str).to_numpy()
X = cluster_features.to_numpy(dtype=float)

print("Cluster feature matrix:", X.shape, "(clusters x technologies)")

# 2) Size proxy (use total salience mass across technologies)
size_vals = cluster_features.sum(axis=1).to_numpy()
size_scaled = 30 + 200 * (np.sqrt(size_vals) / (np.sqrt(size_vals).max() + 1e-9))

# 3) Load framing lens mapping if available (optional)
lens_map = None
json_path = os.path.join(ARTIFACT_DIR, "framing_lens_mappings.json")
if os.path.exists(json_path):
    with open(json_path, "r") as f:
        lens_map = json.load(f)
    # Normalize to {cluster_label: lens}
    if lens_map and all(isinstance(v, list) for v in lens_map.values()):
        inv = {}
        for lens, clist in lens_map.items():
            for c in clist:
                inv[str(c)] = str(lens)
        lens_map = inv
    else:
        lens_map = {str(k): str(v) for k, v in lens_map.items()}

if lens_map is None:
    lens_labels = np.array(["(no lens)"] * len(clusters), dtype=object)
else:
    lens_labels = np.array([lens_map.get(c, "(unmapped)") for c in clusters], dtype=object)

# Encode lenses to integer codes
unique_lenses = pd.unique(lens_labels)
lens_to_code = {lens: i for i, lens in enumerate(unique_lenses)}
codes = np.array([lens_to_code[l] for l in lens_labels], dtype=int)

# 4) PCA projection (standardize across technologies)
X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True) + 1e-9
Xz = (X - X_mean) / X_std

from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=0)
coords = pca.fit_transform(Xz)
x, y = coords[:, 0], coords[:, 1]
expl = pca.explained_variance_ratio_

# 5) Plot
plt.figure(figsize=(10, 7))
ax = plt.gca()

# Use a categorical colormap; map integer codes -> colors
cmap = plt.get_cmap("tab20", len(unique_lenses))  # tab20 gives distinct-ish categories
sc = ax.scatter(x, y, s=size_scaled, c=codes, cmap=cmap, alpha=0.85)

ax.set_xlabel(f"Concern space (PC1; {expl[0]:.0%} var)")
ax.set_ylabel(f"Concern space (PC2; {expl[1]:.0%} var)")
ax.set_title("Figure: Concern Space of Public Technology Concerns\n(Clusters projected by technology-salience profile; size ~ frequency proxy)")

# Annotate top-N largest clusters (by size proxy)
topN = 12
top_idx = np.argsort(size_vals)[::-1][:topN]
for i in top_idx:
    ax.annotate(clusters[i], (x[i], y[i]), textcoords="offset points", xytext=(6, 4), ha="left", fontsize=9)

# Legend: show up to 12 most frequent lenses (or all if <= 12)
lens_counts = pd.Series(lens_labels).value_counts()
top_lenses = lens_counts.index[:12].tolist() if len(lens_counts) > 12 else lens_counts.index.tolist()

legend_handles = [
    Line2D([0], [0], marker='o', linestyle='', markersize=8,
           markerfacecolor=cmap(lens_to_code[lens]), markeredgecolor='none', label=lens)
    for lens in top_lenses
]
ax.legend(handles=legend_handles, title="Framing lens (top shown)", loc="best", frameon=True)

plt.tight_layout()
out_path = "figure_concern_space_pca.png"
plt.savefig(out_path, dpi=200)
plt.show()

print("Saved figure to:", out_path)


This cell calculates the prevalence of various benefit framing lenses across the analyzed public dialogue documents by aggregating the number of benefit phrases associated with each lens. This analysis is crucial as it provides insights into how different benefits are emphasized in public discussions about technology, allowing researchers to identify dominant narratives and potential gaps in public understanding or concern. The visual representation of this data further aids in interpreting the relative importance of each framing lens, informing future dialogue strategies and policy recommendations.

In [ ]:
# @title Compute benefit framing lens prevalence

show_status("Computing benefit framing lens prevalence...")

lens_prevalence = {}
for lens_name, data in BENEFIT_FRAMING_LENS_MAPPINGS.items():
    lens_mask = benefits_df['cluster_id'].isin(data['cluster_ids'])
    lens_prevalence[lens_name] = lens_mask.sum()

lens_prev_df = pd.DataFrame([
    {'Framing Lens': k, 'Benefits': v}
    for k, v in lens_prevalence.items()
]).sort_values('Benefits', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(lens_prev_df['Framing Lens'], lens_prev_df['Benefits'], color='steelblue')
ax.set_xlabel('Number of Benefit Phrases')
ax.set_title('Benefit Framing Lens Prevalence Across All Documents')

for bar, val in zip(bars, lens_prev_df['Benefits']):
    ax.text(val + 10, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_framing_lens_prevalence.png", dpi=150, bbox_inches='tight')
plt.show()

show_complete("Benefit framing lens prevalence computed")


This cell generates a dendrogram to visualize the hierarchical relationships among benefit clusters derived from the public dialogue documents. By comparing the clustering of these benefits with the predefined framing lenses, the analysis assesses the validity of the lens scheme; a strong alignment indicates that the lenses are well-supported by the underlying data structure, while significant divergence could reveal new insights into public perceptions of technology benefits. This step is crucial for understanding how shared concerns and benefits are framed across different technologies, ultimately informing more effective communication strategies in public dialogue.

In [ ]:
# @title Hierarchical structure of benefit clusters (dendrogram)
# Validates the framing-lens scheme by showing whether benefit clusters
# that the LLM grouped into the same lens also cluster together in a
# data-driven hierarchy. Strong correspondence supports the lens scheme;
# divergence is a finding worth reporting.

import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

show_status("Computing benefit-cluster dendrogram...")

# L2-normalise benefit centroids (cosine distance via euclidean on normed vectors)
_norm = np.linalg.norm(benefit_centroids, axis=1, keepdims=True)
benefit_centroids_normed = benefit_centroids / np.clip(_norm, 1e-12, None)

# Pairwise cosine distance and Ward linkage
distances = pdist(benefit_centroids_normed, metric="cosine")
Z = linkage(distances, method="average")  # average linkage works well with cosine

# Build leaf labels using existing benefit cluster labels
benefit_cluster_label_lookup = {
    int(k): v.get("label", f"Cluster {k}")
    for k, v in benefit_cluster_labels_dict.items()
}
leaf_labels = [benefit_cluster_label_lookup.get(i, f"Cluster {i}")
               for i in range(len(benefit_centroids_normed))]

# Build cluster->lens lookup so we can colour leaves by lens
benefit_cluster_to_lens = {}
for lens_name, info in BENEFIT_FRAMING_LENS_MAPPINGS.items():
    for cid in info.get("cluster_ids", []):
        benefit_cluster_to_lens[int(cid)] = lens_name

# Assign a colour per lens
import matplotlib.cm as cm
lens_names_b = list(BENEFIT_FRAMING_LENS_MAPPINGS.keys())
lens_colours_b = {name: cm.tab20(i / max(len(lens_names_b), 1))
                  for i, name in enumerate(lens_names_b)}

# Plot
fig, ax = plt.subplots(figsize=(11, max(8, len(leaf_labels) * 0.18)))
dd = dendrogram(
    Z,
    labels=leaf_labels,
    orientation="left",
    leaf_font_size=8,
    color_threshold=0,
    above_threshold_color="grey",
    ax=ax,
)
# Recolour leaf labels by lens
ylabels = ax.get_ymajorticklabels()
for label in ylabels:
    cid_label = label.get_text()
    matched_cid = next((cid for cid, name in benefit_cluster_label_lookup.items()
                        if name == cid_label), None)
    if matched_cid is not None:
        lens = benefit_cluster_to_lens.get(matched_cid, None)
        if lens is not None:
            label.set_color(lens_colours_b.get(lens, "black"))

ax.set_xlabel("Cosine distance between benefit cluster centroids")
ax.set_title("Hierarchical structure of benefit clusters\n(leaf colour = LLM-assigned framing lens)")

# Legend
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=col, label=name) for name, col in lens_colours_b.items()]
ax.legend(handles=legend_handles, loc="lower right", fontsize=7, title="Framing lens")

plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_cluster_dendrogram.png", dpi=200, bbox_inches="tight")
plt.show()

# Quantitative comparison: cut the dendrogram at the same number of branches
# as the lens scheme, then compute Adjusted Rand Index against the lens
# assignment. If high, the lenses are well-supported by the data structure.
from sklearn.metrics import adjusted_rand_score

n_lenses_b = len(BENEFIT_FRAMING_LENS_MAPPINGS)
data_driven_groups = fcluster(Z, t=n_lenses_b, criterion="maxclust")
lens_assignment = np.array([
    lens_names_b.index(benefit_cluster_to_lens.get(i, lens_names_b[0]))
    if benefit_cluster_to_lens.get(i) in lens_names_b else -1
    for i in range(len(benefit_centroids_normed))
])
mask = lens_assignment >= 0
ari_b = adjusted_rand_score(lens_assignment[mask], data_driven_groups[mask])
print(f"\nNumber of benefit lenses (LLM-derived): {n_lenses_b}")
print(f"Adjusted Rand Index between lens assignment and dendrogram cut at "
      f"{n_lenses_b} groups: {ari_b:.3f}")
print("Higher = stronger correspondence between data-driven hierarchy and "
      "LLM-derived lens scheme.")

show_complete("Benefit-cluster dendrogram complete.")

This code snippet prepares the dataset for analyzing shared benefits across various technologies by ensuring that the benefits data frame includes a metadata column for technology. It also identifies cross-cutting clusters with a normalized entropy above a specified threshold, which is crucial for understanding the diversity of benefits associated with different technologies in the public dialogue documents. This step is essential for the subsequent analysis of how these benefits are perceived and framed in relation to overarching themes in public discourse.

In [ ]:
# --- Section 6 prerequisites (canonical tech + cross-cutting clusters) ---

TECH_COL = "technology_meta"

# Ensure benefits_df has technology_meta
if TECH_COL not in benefits_df.columns:
    benefits_df = benefits_df.merge(
        chunks_df[["chunk_id", TECH_COL]],
        on="chunk_id",
        how="left"
    )

CROSS_CUTTING_ENTROPY_THRESHOLD = 0.5
cross_cutting_cluster_ids_benefits = {
    cid for cid, ent in normalized_entropy_benefits.items()
    if ent >= CROSS_CUTTING_ENTROPY_THRESHOLD
}


This cell computes and visualizes the shared benefits associated with various technologies by analyzing the prevalence and diversity of benefit-related phrases across 65 public dialogue documents. By identifying the top clusters of benefits that exhibit both high prevalence and cross-technology spread, this analysis highlights common themes that resonate across different technologies, which is crucial for understanding public perceptions and informing policy discussions in the context of emerging technologies. The resulting data not only aids in framing the dialogue around shared societal benefits but also enhances the overall understanding of stakeholder concerns in public discourse.

In [ ]:
# @title Shared benefits across technologies (document-weighted)

show_status("Computing shared benefit structure (all technologies)...")

TECH_COL = "technology_meta"

# Canonical list of technologies from metadata
technologies = sorted(benefits_df[TECH_COL].dropna().unique().tolist())

benefit_salience_rows = {}
for tech in technologies:
    tech_mask = benefits_df[TECH_COL] == tech
    tech_total = int(tech_mask.sum())
    if tech_total == 0:
        continue
    counts = benefits_df.loc[tech_mask, "cluster_id"].value_counts()
    benefit_salience_rows[tech] = (counts / tech_total).to_dict()

benefit_salience_df = pd.DataFrame(benefit_salience_rows).T.fillna(0)

# Document-weighted global prevalence of clusters
benefit_global_cluster_prevalence = benefits_df["cluster_id"].value_counts(normalize=True)

benefit_cluster_meta_df = (
    pd.DataFrame(
        {
            "cluster_id": list(benefit_cluster_entropy.keys()),
            "tech_entropy": [benefit_cluster_entropy[c] for c in benefit_cluster_entropy.keys()],
            "global_prevalence": [benefit_global_cluster_prevalence.get(c, 0) for c in benefit_cluster_entropy.keys()],
        }
    )
    .set_index("cluster_id")
    .sort_values(["global_prevalence", "tech_entropy"], ascending=False)
)

# Shared benefits = high prevalence + high entropy
shared_benefits = benefit_cluster_meta_df.head(20)

print("Top shared benefit clusters (high prevalence + cross-technology spread):")
display(shared_benefits.head(12))

plt.figure(figsize=(10, 5))
shared_benefits.sort_values("global_prevalence")["global_prevalence"].plot(kind="barh", legend=False)
plt.title("Shared benefits across technologies (top 20 clusters by prevalence x spread)")
plt.xlabel("Share of all extracted benefit phrases")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "shared_benefits_top20.png")
plt.show()

shared_benefits.reset_index().to_csv(OUTPUT_FOLDER / "shared_benefits_top20.csv", index=False)

show_complete("Shared benefits computed")


This cell computes and visualizes the prevalence and diversity of shared benefit framings across various technologies by analyzing their distribution in the dataset. By calculating the overall prevalence and entropy of each framing lens, it identifies which benefits are most commonly articulated and how they vary across technologies, providing insights into public perceptions and concerns. This analysis is crucial for understanding the common themes in public dialogue, which can inform policy-making and communication strategies in technology development.

In [ ]:
# @title Shared benefit framings across technologies (document-weighted)

from scipy.stats import entropy as scipy_entropy

show_status("Computing shared benefit framing structure (all technologies)...")

TECH_COL = 'technology_meta'
technologies = sorted(benefits_df[TECH_COL].dropna().unique().tolist())

# Lens prevalence overall + entropy across technologies
benefit_lens_stats = []
for lens_name, data in BENEFIT_FRAMING_LENS_MAPPINGS.items():
    lens_clusters = set(data['cluster_ids'])
    lens_mask = benefits_df['cluster_id'].isin(lens_clusters)
    overall_prev = lens_mask.mean()

    tech_counts = benefits_df.loc[lens_mask, TECH_COL].value_counts()
    tech_probs = (tech_counts / tech_counts.sum()).values if tech_counts.sum() > 0 else np.array([])
    lens_entropy = float(scipy_entropy(tech_probs)) if len(tech_probs) > 1 else 0.0

    benefit_lens_stats.append({
        'framing_lens': lens_name,
        'overall_prevalence': float(overall_prev),
        'tech_entropy': lens_entropy,
        'n_benefits_in_lens': int(lens_mask.sum())
    })

benefit_lens_meta_df = (pd.DataFrame(benefit_lens_stats)
                        .sort_values(['overall_prevalence','tech_entropy'], ascending=False))

print("Shared benefit framings (high prevalence + cross-technology spread):")
display(benefit_lens_meta_df.head(12))

plt.figure(figsize=(7,5))
plt.scatter(benefit_lens_meta_df['tech_entropy'], benefit_lens_meta_df['overall_prevalence'])
for _, r in benefit_lens_meta_df.iterrows():
    plt.text(r['tech_entropy'], r['overall_prevalence'], r['framing_lens'], fontsize=8)
plt.xlabel("Entropy across technologies")
plt.ylabel("Share of all extracted benefits")
plt.title("Shared benefit framings across technologies")
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "shared_benefit_framings_scatter.png")
plt.show()

benefit_lens_meta_df.to_csv(OUTPUT_FOLDER / "shared_benefit_framings.csv", index=False)
show_complete("Shared benefit framings computed")


This cell visualizes the stable core of public technology benefits by plotting the relationship between the entropy of benefit clusters and their frequency across the analyzed documents. By identifying clusters that are both cross-cutting and frequently mentioned, this analysis highlights key areas of shared concern and benefit that are crucial for understanding public sentiment and guiding future technology policy discussions. The resulting scatter plot aids in discerning which benefits resonate most broadly across different technologies, thereby informing stakeholders about the most relevant themes in public dialogue.

In [ ]:
# @title Visualise the stable core of public technology benefits

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# ---- prerequisites ----
if "benefit_cluster_entropy" not in globals():
    raise NameError("benefit_cluster_entropy not found. Run cell 59 first.")
if "benefits_df" not in globals():
    raise NameError("benefits_df not found. Run benefit extraction first.")

TECH_COL = "technology_meta"
ENTROPY_THRESHOLD = 0.5  # canonical cross-cutting threshold

# ---- build cluster-level dataframe ----
cluster_size = benefits_df["cluster_id"].value_counts()

# Note: benefit_cluster_entropy stores RAW entropy. We need normalized entropy
# for the 0.5 threshold to be meaningful. Use normalized_entropy_benefits.
df = (
    pd.DataFrame({
        "cluster_id": list(normalized_entropy_benefits.keys()),
        "entropy": [normalized_entropy_benefits[c] for c in normalized_entropy_benefits.keys()],
    })
    .set_index("cluster_id")
)

df["size"] = cluster_size
df["size"] = df["size"].fillna(0)
df = df[df["size"] > 0]

if "BENEFIT_CLUSTER_LABELS" in globals():
    df["label"] = df.index.map(lambda c: BENEFIT_CLUSTER_LABELS.get(c, f"Cluster {c}"))
else:
    df["label"] = df.index.map(lambda c: f"Cluster {c}")

entropy_vals = df["entropy"].to_numpy(dtype=float)
size = df["size"].to_numpy(dtype=float)
labels = df["label"].to_numpy(dtype=str)

# ---- define cross-cutting + stable core ----
cross_cutting = entropy_vals >= ENTROPY_THRESHOLD
size_thresh = np.quantile(size, 0.75)
stable_core = cross_cutting & (size >= size_thresh)

score = entropy_vals * np.log1p(size)
annotate_idx = np.argsort(score)[::-1][:12]

plt.figure(figsize=(10, 7))
ax = plt.gca()

core_region = Rectangle(
    (ENTROPY_THRESHOLD, size_thresh),
    1.0 - ENTROPY_THRESHOLD,
    size.max() * 1.05 - size_thresh,
    alpha=0.12,
    zorder=0
)
ax.add_patch(core_region)

ax.text(
    ENTROPY_THRESHOLD + 0.01,
    size.max() * 1.02,
    "Stable core\n(cross-cutting + high frequency)",
    fontsize=10,
    va="top"
)

plt.scatter(entropy_vals[~cross_cutting], size[~cross_cutting], alpha=0.6, s=35, label="More tech-specific clusters")
plt.scatter(entropy_vals[cross_cutting], size[cross_cutting], alpha=0.8, s=45, label="More cross-cutting clusters")
plt.scatter(entropy_vals[stable_core], size[stable_core], s=90, label="Stable core")

for i in annotate_idx:
    plt.annotate(labels[i], (entropy_vals[i], size[i]), textcoords="offset points", xytext=(6, 4), ha="left", fontsize=9)

plt.axvline(ENTROPY_THRESHOLD, linestyle="--", linewidth=1)
plt.axhline(size_thresh, linestyle="--", linewidth=1)

plt.xlabel("Technology-distribution entropy (higher = more cross-cutting)")
plt.ylabel("Cluster frequency / size (higher = more recurrent)")
plt.title("Figure: The Stable Core of Public Technology Benefits")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_FOLDER / "benefit_stable_core_scatter.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()


This cell visualizes the benefit embedding space derived from the analysis of public dialogue documents, employing PCA to reduce the dimensionality of the technology-salience profiles. By plotting the clusters of technologies in a two-dimensional space, it highlights the relationships and salience of various benefits, while also incorporating framing lenses to provide context for the observed patterns. This visualization is crucial for understanding how different technologies are perceived in terms of shared benefits, which can inform future dialogue and policy-making in public engagement on technological advancements.

In [ ]:
# @title Visualise the benefit embedding space

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ARTIFACT_DIR = globals().get("ARTIFACT_DIR", "analysis_artifacts")
EXCEL_PATH = globals().get("EXCEL_PATH", os.path.join(ARTIFACT_DIR, "public_dialogue_analysis_v9.xlsx"))

if "benefit_salience_df" not in globals():
    raise NameError("benefit_salience_df not found. Run Section 6.1a (benefits) to compute it first.")

tech_by_cluster = benefit_salience_df.copy()
tech_by_cluster = tech_by_cluster.apply(pd.to_numeric, errors="coerce").fillna(0)

# Convert to clusters x technologies
cluster_features = tech_by_cluster.T
clusters = cluster_features.index.astype(str).to_numpy()
X = cluster_features.to_numpy(dtype=float)

print("Cluster feature matrix:", X.shape, "(clusters x technologies)")

size_vals = cluster_features.sum(axis=1).to_numpy()
size_scaled = 30 + 200 * (np.sqrt(size_vals) / (np.sqrt(size_vals).max() + 1e-9))

# Load benefit framing lens mapping if available
lens_map = None
if "BENEFIT_FRAMING_LENS_MAPPINGS" in globals():
    lens_map_raw = BENEFIT_FRAMING_LENS_MAPPINGS
    lens_map = {}
    for lens_name, data in lens_map_raw.items():
        for cid in data.get('cluster_ids', []):
            lens_map[str(cid)] = str(lens_name)
else:
    json_path = os.path.join(ARTIFACT_DIR, "benefit_framing_lens_mappings.json")
    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            lens_map = json.load(f)
        if lens_map and all(isinstance(v, list) for v in lens_map.values()):
            inv = {}
            for lens, clist in lens_map.items():
                for c in clist:
                    inv[str(c)] = str(lens)
            lens_map = inv
        else:
            lens_map = {str(k): str(v) for k, v in lens_map.items()}

if lens_map is None:
    lens_labels = np.array(["(no lens)"] * len(clusters), dtype=object)
else:
    lens_labels = np.array([lens_map.get(c, "(unmapped)") for c in clusters], dtype=object)

unique_lenses = pd.unique(lens_labels)
lens_to_code = {lens: i for i, lens in enumerate(unique_lenses)}
codes = np.array([lens_to_code[l] for l in lens_labels], dtype=int)

# PCA projection
X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True) + 1e-9
Xz = (X - X_mean) / X_std

from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=0)
coords = pca.fit_transform(Xz)
x, y = coords[:, 0], coords[:, 1]
expl = pca.explained_variance_ratio_

plt.figure(figsize=(10, 7))
ax = plt.gca()

cmap = plt.get_cmap("tab20", len(unique_lenses))
sc = ax.scatter(x, y, s=size_scaled, c=codes, cmap=cmap, alpha=0.85)

ax.set_xlabel(f"Benefit space (PC1; {expl[0]:.0%} var)")
ax.set_ylabel(f"Benefit space (PC2; {expl[1]:.0%} var)")
ax.set_title("Figure: Benefit Space of Public Technology Benefits\n(Clusters projected by technology-salience profile; size ~ frequency proxy)")

topN = 12
top_idx = np.argsort(size_vals)[::-1][:topN]
for i in top_idx:
    ax.annotate(clusters[i], (x[i], y[i]), textcoords="offset points", xytext=(6, 4), ha="left", fontsize=9)

lens_counts = pd.Series(lens_labels).value_counts()
top_lenses = lens_counts.index[:12].tolist() if len(lens_counts) > 12 else lens_counts.index.tolist()

legend_handles = [
    Line2D([0], [0], marker='o', linestyle='', markersize=8,
           markerfacecolor=cmap(lens_to_code[lens]), markeredgecolor='none', label=lens)
    for lens in top_lenses
]
ax.legend(handles=legend_handles, title="Framing lens (top shown)", loc="best", frameon=True)

plt.tight_layout()
out_path = "figure_benefit_space_pca.png"
plt.savefig(out_path, dpi=200)
plt.show()

print("Saved figure to:", out_path)


This cell imports a predefined volume table from the dialogue_utils module, which is essential for analyzing the shared concern and benefit structures across the public dialogue documents. By leveraging this data, the analysis can identify cross-cutting themes and develop document-weighted framing lens profiles, ultimately enhancing the understanding of public sentiment and discourse surrounding various technologies.

In [ ]:
# _volume_table — imported from dialogue_utils (see Cell 5)
pass

This cell imports the `_top_clusters` variable from the `dialogue_utils` module, which is essential for analyzing the shared concern and benefit structures across various technologies in the public dialogue documents. By leveraging these clusters, the analysis can identify prevalent themes and framing profiles, ultimately enhancing the understanding of public sentiment and priorities regarding technological advancements in the UK.

In [ ]:
# _top_clusters — imported from dialogue_utils (see Cell 5)
pass